# Kiểm thử yêu cầu phi chức năng Pet Healthcare trên Kaggle CPU

Chọn **Accelerator None** và bật **Internet**. Notebook này chạy ứng dụng Node và PostgreSQL trên database thử mới, không dùng dữ liệu vận hành.

| Yêu cầu | Bài kiểm tra |
|---|---|
| PCN01 | Ít nhất 95% yêu cầu thành công trong 2 giây, kiểm riêng từng API đọc, lọc và ghi |
| PCN02 | Đạt 100 người dùng, làm nóng thêm 30 giây, sau đó đo đủ **1800 giây**, tỷ lệ lỗi không quá 1% |
| PCN03 | Kiểm mật khẩu băm của toàn bộ tài khoản thử và ma trận truy cập trái quyền |
| PCN04 | Đặt lịch thật trong 5 thao tác trên giao diện, không tính đăng nhập hoặc từng phím gõ |
| PCN05 | Chrome và Edge thật, ở 390 px và 1366 px, kiểm 12 trang chính và nút thao tác |
| PCN06 | Tạo 20 hồ sơ thú cưng qua API; khởi động lại cả API và PostgreSQL; so sánh tất cả trường |
| PCN07 | Tạo 8 bản sao, giữ 7 bản mới nhất, khôi phục vào DB mới trong 30 phút; **chưa chứng minh lịch sao lưu hằng ngày** |

Kết quả có PASS_IN_SCOPE, FAIL, PARTIAL và NOT_RUN. Không gọi toàn bộ chương trình đạt chỉ vì bài tải đạt. Khoảng 30 phút đo tải, cộng thời gian cài đặt, kiểm tra và sao lưu. Chạy cell xuất kết quả cả khi có lỗi.

In [ ]:
from pathlib import Path
import hashlib, json, os, platform, pwd, re, shutil, signal, subprocess, sys, urllib.request, uuid
from datetime import datetime, timezone

WORKING = Path('/kaggle/working')
assert WORKING.is_dir(), 'Notebook đầy đủ cần chạy trên Kaggle.'
REPO_URL = 'https://github.com/Nguyenquanghong/Pet-healthcare.git'
COMMIT_SHA = 'DIEN_SHA_40_KY_TU_CUA_BAN_DA_CO_BO_KIEM_THU_PCN'
SKIP_BROWSER = False  # Đổi thành True chỉ khi chấp nhận PCN04 và PCN05 là NOT_RUN.
PROTOCOL = 'pet-healthcare-pcn-v1'
repo = WORKING / 'pet-healthcare-pcn'
assert re.fullmatch(r'[0-9a-fA-F]{40}', COMMIT_SHA), 'Điền SHA có đủ notebook và các file pcn_* trước khi chạy.'

## Lấy source đúng phiên bản

Các file mới phải có trong commit đã được tải lên repository trước khi chọn SHA. Notebook không tự commit hoặc push. Checkout đang có thay đổi sẽ bị từ chối, không bị ghi đè.

In [ ]:
if not repo.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(repo)], check=True)
dirty = subprocess.check_output(['git', 'status', '--porcelain'], cwd=repo, text=True).strip()
assert not dirty, 'Checkout đang có thay đổi; kiểm tra trước khi tiếp tục.'
subprocess.run(['git', 'fetch', 'origin', COMMIT_SHA], cwd=repo, check=True)
subprocess.run(['git', 'checkout', '--detach', COMMIT_SHA], cwd=repo, check=True)
actual = subprocess.check_output(['git', 'rev-parse', 'HEAD'], cwd=repo, text=True).strip()
assert actual.lower() == COMMIT_SHA.lower()
sys.path.insert(0, str(repo / 'benchmarks/kaggle'))
from pcn_contract import PROTOCOL as source_protocol
assert source_protocol == PROTOCOL, 'SHA này chưa có bộ kiểm thử PCN đúng phiên bản.'
print('Source:', actual, '| protocol:', source_protocol)

## Cài công cụ và kiểm tra bộ đánh giá
Dùng môi trường CPU và Node được ghi phiên bản trong manifest. Browser được cài trong phiên Kaggle, không liên quan trình duyệt trên máy cá nhân.

In [ ]:
def find_pg_bin():
    candidates = []
    if shutil.which("pg_config"):
        candidates.append(Path(subprocess.check_output(["pg_config", "--bindir"], text=True).strip()))
    candidates.extend(sorted(Path("/usr/lib/postgresql").glob("*/bin"), reverse=True))
    return next((p for p in candidates if all((p / tool).is_file() for tool in ("initdb", "pg_ctl", "postgres", "psql", "createdb"))), None)

pg_bin = find_pg_bin()
if pg_bin is None or not shutil.which("pg_config"):
    assert os.geteuid() == 0, "Không có quyền cài PostgreSQL server trong phiên này."
    subprocess.run(["apt-get", "update"], check=True)
    subprocess.run(["apt-get", "install", "-y", "postgresql", "postgresql-client", "libpq-dev"], check=True)
    pg_bin = find_pg_bin()
assert pg_bin is not None, "Chưa tìm được PostgreSQL server; xem output cài đặt."
os.environ["BENCH_PG_BIN"] = str(pg_bin)
os.environ["PATH"] = str(pg_bin) + os.pathsep + os.environ["PATH"]
if os.geteuid() == 0:
    assert shutil.which("runuser"), "Thiếu runuser để chạy PostgreSQL bằng user postgres."
    pwd.getpwnam("postgres")

assert platform.machine() in ("x86_64", "amd64"), "Bản Node pin này dành cho Kaggle Linux x64."
node_dir = WORKING / "node24"
if (node_dir / "bin/node").is_file():
    os.environ["PATH"] = str(node_dir / "bin") + os.pathsep + os.environ["PATH"]
node_ok = shutil.which("node") and subprocess.check_output(["node", "--version"], text=True).strip().startswith("v24.")
if not node_ok:
    version = "v24.15.0"
    name = f"node-{version}-linux-x64.tar.xz"
    base = f"https://nodejs.org/dist/{version}/"
    archive = WORKING / name
    urllib.request.urlretrieve(base + name, archive)
    checksums = urllib.request.urlopen(base + "SHASUMS256.txt", timeout=60).read().decode()
    expected = next(line.split()[0] for line in checksums.splitlines() if line.split()[-1] == name)
    assert hashlib.sha256(archive.read_bytes()).hexdigest() == expected, "Node checksum không khớp."
    node_dir.mkdir(exist_ok=True)
    subprocess.run(["tar", "-xJf", str(archive), "-C", str(node_dir), "--strip-components=1"], check=True)
    os.environ["PATH"] = str(node_dir / "bin") + os.pathsep + os.environ["PATH"]
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(repo / "benchmarks/kaggle/requirements.txt")], check=True)
for program, args in {"node": ["--version"], "npm": ["--version"], "pg_config": ["--version"]}.items():
    print(program, subprocess.check_output([program, *args], text=True).strip())
print("PostgreSQL server:", subprocess.check_output([str(pg_bin / "postgres"), "--version"], text=True).strip())
print("Python:", sys.version.split()[0], "| Locust:", subprocess.check_output([sys.executable, "-m", "locust", "--version"], text=True).strip())
subprocess.run([sys.executable, "-m", "unittest", "discover", "-s", str(repo / "benchmarks/kaggle"), "-p", "test_*.py", "-v"], check=True)

## Chạy kiểm thử và tự xuất bằng chứng

Không có tuỳ chọn rút ngắn cửa sổ 1800 giây trong notebook này. Bài thử tạo dữ liệu giả và database riêng. Browser chạy sau tải để không làm sai phép đo API. Nếu cài Chrome hoặc Edge thất bại, kết quả giao diện là NOT_RUN.

In [ ]:
from pcn_export import export_evidence
run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:8]
evidence = WORKING / ('pcn-evidence-' + run_id)
runner_log = WORKING / ('pcn-runner-' + run_id + '.log')
args = [sys.executable, str(repo / 'benchmarks/kaggle/pcn_runner.py'), '--repo', str(repo), '--output', str(evidence)]
if SKIP_BROWSER:
    args.append('--skip-browser')
process = None
try:
    with runner_log.open('w', encoding='utf-8') as log:
        process = subprocess.Popen(args, cwd=repo, stdout=log, stderr=subprocess.STDOUT)
        try:
            code = process.wait()
            print('Runner exit:', code)
        except KeyboardInterrupt:
            process.send_signal(signal.SIGINT)
            try:
                process.wait(timeout=60)
            except subprocess.TimeoutExpired:
                process.terminate()
                process.wait(timeout=20)
            print('Đã dừng. Kết quả một phần sẽ được giữ lại.')
finally:
    evidence.mkdir(parents=True, exist_ok=True)
    if runner_log.exists():
        shutil.copyfile(runner_log, evidence / 'runner.log')
    archive = export_evidence(evidence, WORKING)
    print('ZIP bằng chứng:', archive)
report_file = evidence / 'pcn-report.json'
if report_file.exists():
    summary = json.loads(report_file.read_text(encoding='utf-8'))
    for key, result in summary['requirements'].items():
        print(key, result['status'], '|', result.get('scope', ''))
    print('Kết luận:', summary['overall'])
else:
    print('PARTIAL_OR_FAILED: chưa có báo cáo cuối. Xem runner.log trong ZIP.')

## Xuất lại sau khi có lỗi hoặc bị ngắt

Tải file ZIP ở mục **Output**. ZIP gồm báo cáo JSON/CSV, request thô, tài nguyên, log và ảnh giao diện. Không chứa JWT fixture, ảnh chụp dữ liệu trước restart hoặc database dump riêng.

PASS_IN_SCOPE xác nhận các tình huống đã liệt kê. PCN07 chỉ có thể là PARTIAL trong một phiên chạy: cần thêm log công việc sao lưu hằng ngày trong ít nhất 7 ngày để nghiệm thu đầy đủ.

In [ ]:
assert 'evidence' in globals(), 'Chạy cell kiểm thử trước để có đường dẫn bằng chứng.'
archive = export_evidence(evidence, WORKING)
print('ZIP mới:', archive)
print('Báo cáo:', evidence / 'pcn-report.csv')